In [ ]:
!pip install -q torch==2.4.1 triton==3.0.0

In [ ]:
import torch
import triton
import triton.language as tl

## Preprocessing

In [ ]:
def pack_blocks(mat: torch.Tensor, block_size: int):
    """
    Vectorized pack for EW N:M sparsity along reduction dim K (per row).

    - Assumes mat is 2D with shape (M, K).
    - Enforces blocks of `block_size` contiguous weights along K.
    - If K is not divisible by block_size, the row is zero-padded to keep
      block alignment (padded blocks will have 0 nonzeros).

    Returns:
      vals:         concatenated non-zeros
      idx_in_block: positions within each block (0..block_size-1)
      block_ptr:    prefix-sum into vals per block (len = n_blocks + 1)
      block_coords: (row, k_start) for each block to locate it in mat
    """
    if mat.dim() != 2:
        raise ValueError("pack_blocks expects a 2D tensor (M, K)")

    M, K = mat.shape
    pad = (block_size - K % block_size) % block_size
    if pad:
        mat = torch.nn.functional.pad(mat, (0, pad))
        K = mat.shape[1]

    num_blocks = K // block_size

    # shape: (M, num_blocks, block_size) -> (M*num_blocks, block_size)
    blocks_flat = mat.view(M, num_blocks, block_size).reshape(-1, block_size)

    mask = blocks_flat != 0
    counts = mask.sum(dim=1)  # per-block NNZ
    block_ptr = torch.cat([
        torch.zeros(1, device=mat.device, dtype=torch.long),
        counts.cumsum(dim=0)
    ], dim=0)

    nz = mask.nonzero(as_tuple=False)
    block_idx = nz[:, 0]
    idx_in_block = nz[:, 1]
    vals = blocks_flat[block_idx, idx_in_block]

    block_rows = torch.arange(M, device=mat.device).unsqueeze(1).expand(M, num_blocks).reshape(-1)
    block_kstart = (torch.arange(num_blocks, device=mat.device) * block_size).unsqueeze(0).expand(M, num_blocks).reshape(-1)
    block_coords = torch.stack([block_rows, block_kstart], dim=1)

    return vals, idx_in_block, block_ptr, block_coords

# Call this after defining your matrix, e.g. see the next cell.

In [8]:
# Example: 2:4 EW-N:M sparse matrix along reduction dim K (block_size=4)
# Each block of 4 contiguous entries *per row* has exactly 2 non-zeros.
A = torch.tensor([
    # blocks: [0..3] , [4..7]
    [5, 0, 7, 0,   0, 2, 0, 3],
    [0, 4, 0, 6,   8, 0, 1, 0],
    [9, 0, 0, 10,  0, 11, 12, 0],
    [0, 13, 14, 0,  15, 0, 0, 16],
], dtype=torch.float32)

block_size = 4  # matches the 2:4 pattern used in SparTA-style packing
vals, idx_in_block, block_ptr, block_coords = pack_blocks(A, block_size=block_size)
print("vals:", vals)                         # nonzeros in block order
print("idx_in_block:", idx_in_block)           # positions 0..3 within each block
print("block_ptr:", block_ptr)                 # block b -> vals[block_ptr[b]:block_ptr[b+1]]
print("block_coords (row, k_start):", block_coords)

# quick sanity check for EW 2:4 sparsity (along K)
block_counts = block_ptr[1:] - block_ptr[:-1]
assert (block_counts == 2).all(), "Matrix violates EW 2:4 sparsity along K"
print("nonzeros per block:", block_counts)

vals: tensor([ 5.,  7.,  2.,  3.,  4.,  6.,  8.,  1.,  9., 10., 11., 12., 13., 14.,
        15., 16.])
idx_in_block: tensor([0, 2, 1, 3, 1, 3, 0, 2, 0, 3, 1, 2, 1, 2, 0, 3])
block_ptr: tensor([ 0,  2,  4,  6,  8, 10, 12, 14, 16])
block_coords (row, k_start): tensor([[0, 0],
        [0, 4],
        [1, 0],
        [1, 4],
        [2, 0],
        [2, 4],
        [3, 0],
        [3, 4]])
nonzeros per block: tensor([2, 2, 2, 2, 2, 2, 2, 2])
